# Android SMS Classifier — 一键训练（VS Code / Colab）

在 **VS Code** 打开本文件 → 选内核 → **Run All**。

| 内核 | 怎么选 | 要不要上传 zip |
|------|--------|----------------|
| 本机 Python / `.venv` | 右上角 Kernel → 选仓库 `.venv` | **不用**（直接读当前仓库） |
| Google Colab（扩展） | Kernel → Colab / 连接到 Colab GPU | **不用上传**；第 1 格填 `GIT_URL` 自动 clone |

## 合规

- **禁止**把真实短信 / 私有标注推到公网 Git 或上传 Colab。
- 仅合成数据或公司已批准脱敏数据。
- `bert-base-multilingual-cased` 为 Hugging Face【第三方】权重。
- 合成指标不得宣称事务召回 ≥98%。

## VS Code 操作（两种）

**A. 本机跑（你有 RTX 4070，推荐先试）**
1. 安装扩展：Jupyter
2. 打开本 `init.ipynb`
3. 选内核为项目 `.venv`（没有就先 `python -m venv .venv` 并 `pip install -r training/requirements-train.txt`）
4. 下面「配置」格保持 `MODE="auto"`，`GIT_URL=""`
5. Run All

**B. Colab GPU（VS Code Google Colab 扩展）**
1. 安装扩展：Google Colab（或 Jupyter + Colab 内核）
2. 把仓库推到**公司允许的 Git 远端**（不要含真实短信）
3. 打开本 notebook → 内核选 **Colab**，开 GPU
4. 在配置格填 `GIT_URL`（可带 token 的 HTTPS，勿把 token 提交进 Git）
5. Run All（会自动 clone，无需手动上传 zip）

## 0. 配置（一般只改这一格）

In [ ]:
# ========= 用户配置 =========
MODE = "colab"  # auto | local | colab

# 仅 Colab 内核需要：公司 Git HTTPS 地址（合成代码仓即可）
# 例: "https://github.com/ORG/Android_SMS_Classifier.git"
# 私有库临时: "https://<TOKEN>@github.com/ORG/Android_SMS_Classifier.git"  （勿提交 token）
GIT_URL = "https://github.com/xiaobendaoke/Android_SMS_Classifier.git"
GIT_BRANCH = "main"

# Colab clone 目标
COLAB_WORKDIR = "/content/Android_SMS_Classifier"

# 是否在云端重新生成更大合成数据（本机已有 processed 可 False）
REGENERATE_SYNTHETIC = False
PER_LABEL_LANG = 40

# 教师冒烟：>0 只取前 N 条；正式训练保持 0
TEACHER_MAX_SAMPLES = 0

SEED = 42
BERT_MODEL_ID = "google-bert/bert-base-multilingual-cased"  # 【第三方】
# ============================

## 1. 定位项目根目录（自动，无需上传）

In [ ]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path


def running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def find_repo_root(start: Path) -> Path | None:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / "training" / "scripts" / "train_teacher.py").is_file():
            return p
    return None


def clone_repo(url: str, branch: str, dest: Path) -> Path:
    if not url:
        raise ValueError(
            "当前是 Colab 内核，请在配置格填写 GIT_URL（公司允许的远端），"
            "这样会自动 clone，无需手动上传 zip。"
        )
    if dest.exists():
        # 已存在则拉取最新
        if (dest / ".git").exists():
            subprocess.run(["git", "-C", str(dest), "fetch", "--all"], check=False)
            subprocess.run(
                ["git", "-C", str(dest), "checkout", branch], check=False
            )
            subprocess.run(
                ["git", "-C", str(dest), "pull", "--ff-only", "origin", branch],
                check=False,
            )
            return dest
        shutil.rmtree(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--branch", branch, "--depth", "1", url, str(dest)],
        check=True,
    )
    return dest


IN_COLAB = running_in_colab()
effective_mode = MODE
if MODE == "auto":
    effective_mode = "colab" if IN_COLAB else "local"

print(f"IN_COLAB={IN_COLAB} MODE={MODE} -> {effective_mode}")

if effective_mode == "colab":
    PROJECT_ROOT = clone_repo(GIT_URL, GIT_BRANCH, Path(COLAB_WORKDIR))
    BERT_CACHE = Path("/content/hf_cache/bert-base-multilingual-cased")
else:
    PROJECT_ROOT = find_repo_root(Path.cwd())
    if PROJECT_ROOT is None:
        # VS Code 有时 cwd 不在仓库根，再试常见相对位置
        here = Path.cwd()
        for cand in [here, here / "Android_SMS_Classifier", here.parent]:
            PROJECT_ROOT = find_repo_root(cand)
            if PROJECT_ROOT is not None:
                break
    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            "本地模式找不到仓库根目录。请在 VS Code 打开本仓库文件夹后再 Run All。"
        )
    BERT_CACHE = PROJECT_ROOT / "training" / ".cache" / "bert-base-multilingual-cased"

os.chdir(PROJECT_ROOT)
os.environ["PYTHONPATH"] = str(PROJECT_ROOT / "training")
sys.path.insert(0, str(PROJECT_ROOT / "training"))

assert (PROJECT_ROOT / "training" / "scripts" / "train_teacher.py").is_file()
print("PROJECT_ROOT =", PROJECT_ROOT)
print("BERT_CACHE   =", BERT_CACHE)
print("cwd          =", Path.cwd())

try:
    import subprocess as _sp

    print(_sp.check_output(["nvidia-smi", "-L"], text=True, stderr=_sp.STDOUT).strip())
except Exception as exc:  # noqa: BLE001
    print("nvidia-smi:", exc)
print("Python", sys.version)

## 2. 安装依赖

In [ ]:
import subprocess
import sys
from pathlib import Path


def pip_install(args: list[str]) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *args])


pip_install(["-U", "pip"])

lock = Path("training/requirements.lock")
train_req = Path("training/requirements-train.txt")

try:
    if lock.exists():
        pip_install(["-q", "-r", str(lock)])
    if train_req.exists():
        pip_install(["-q", "-r", str(train_req)])
except subprocess.CalledProcessError:
    print("lock/train 安装失败，改用精简依赖…")
    pip_install(
        [
            "-q",
            "tensorflow>=2.16",
            "transformers",
            "tensorflow-model-optimization",
            "scikit-learn",
            "PyYAML",
            "numpy",
        ]
    )

import tensorflow as tf
import transformers

print("TF", tf.__version__)
print("transformers", transformers.__version__)
print("GPU", tf.config.list_physical_devices("GPU"))
if not tf.config.list_physical_devices("GPU"):
    print("WARNING: 未检测到 GPU。本机请装 CUDA 版 TF；Colab 请把 runtime/内核设为 GPU。")

## 3. 准备 / 缓存教师 BERT【第三方】

In [ ]:
from pathlib import Path
import hashlib
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification

BERT_CACHE.mkdir(parents=True, exist_ok=True)
marker = BERT_CACHE / "config.json"

if marker.exists():
    print("Reuse existing BERT cache:", BERT_CACHE)
else:
    print("Downloading", BERT_MODEL_ID, "->", BERT_CACHE)
    tok = AutoTokenizer.from_pretrained(BERT_MODEL_ID)
    mdl = TFAutoModelForSequenceClassification.from_pretrained(
        BERT_MODEL_ID, num_labels=4
    )
    tok.save_pretrained(BERT_CACHE)
    mdl.save_pretrained(BERT_CACHE)
    print("Saved.")


def sha256_dir(path: Path) -> str:
    h = hashlib.sha256()
    for f in sorted(path.rglob("*")):
        if f.is_file():
            h.update(f.relative_to(path).as_posix().encode())
            h.update(f.read_bytes())
    return h.hexdigest()


print("BERT cache sha256:", sha256_dir(BERT_CACHE))

## 4. 数据准备与检查

In [ ]:
import json
import os
import subprocess
import sys
from collections import Counter
from pathlib import Path

os.environ["PYTHONPATH"] = str(PROJECT_ROOT / "training")


def run_py(*script_and_args: str) -> None:
    cmd = [sys.executable, *script_and_args]
    print("+", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(PROJECT_ROOT))


train_jsonl = PROJECT_ROOT / "training" / "data" / "processed" / "train.jsonl"
if REGENERATE_SYNTHETIC or not train_jsonl.exists():
    print("Generating synthetic dataset…")
    run_py(
        "training/scripts/generate_synthetic_dataset.py",
        "--per-label-lang",
        str(PER_LABEL_LANG),
    )
    run_py("training/scripts/build_dataset.py", "--augment-train")
    run_py("training/scripts/build_adversarial_slices.py")
else:
    print("reuse existing processed data:", train_jsonl)

run_py("training/scripts/check_split_leakage.py")
run_py("training/scripts/validate_labels.py")

need = {"TRANSACTION", "AD", "HARASS", "FRAUD"}
print("\n=== split label counts ===")
for split in ["train", "validation", "test"]:
    p = PROJECT_ROOT / "training" / "data" / "processed" / f"{split}.jsonl"
    labels = [
        json.loads(line)["label"]
        for line in p.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    c = Counter(labels)
    print(f"{split}: n={len(labels)} {dict(c)}")
    missing = need - set(c)
    if missing and split != "train":
        print(f"WARNING: {split} missing {sorted(missing)} — 建议 REGENERATE_SYNTHETIC=True 后重跑")

## 5. 微调教师 → 蒸馏 → 剪枝 → 量化 → 验证

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

os.environ["PYTHONPATH"] = str(PROJECT_ROOT / "training")


def run_py(*args: str) -> None:
    cmd = [sys.executable, *args]
    print("\n+", " ".join(cmd), flush=True)
    subprocess.check_call(cmd, cwd=str(PROJECT_ROOT))


teacher_cmd = [
    "training/scripts/train_teacher.py",
    "--model-path",
    str(BERT_CACHE),
    "--seed",
    str(SEED),
]
if TEACHER_MAX_SAMPLES and TEACHER_MAX_SAMPLES > 0:
    teacher_cmd += ["--max-samples", str(TEACHER_MAX_SAMPLES)]

run_py(*teacher_cmd)

logits_manifest = (
    PROJECT_ROOT / "training" / "data" / "manifests" / "teacher_logits_manifest.json"
)
assert logits_manifest.is_file(), "教师失败：缺少 teacher_logits_manifest.json"

run_py("training/scripts/distill_student.py", "--seed", str(SEED))
distill = json.loads(
    (PROJECT_ROOT / "training" / "artifacts" / "student" / "distill_manifest.json").read_text(
        encoding="utf-8"
    )
)
print("distill used_distillation =", distill.get("used_distillation"))
assert distill.get("used_distillation") is True, "未真蒸馏，已中止"

run_py("training/scripts/prune_channels.py", "--seed", str(SEED))
run_py("training/scripts/quantize_int8.py", "--seed", str(SEED))
run_py("training/scripts/verify_tflite.py", "--seed", str(SEED))
run_py("training/scripts/evaluate.py", "--mode", "tflite", "--seed", str(SEED))

tflite = PROJECT_ROOT / "training" / "artifacts" / "student" / "sms_bytecnn_int8.tflite"
assert tflite.is_file(), f"缺少 {tflite}"
print("\nOK pipeline done:", tflite)

## 6. 导出到 Android assets（本机直接写仓库；Colab 则打 zip 下载）

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

os.environ["PYTHONPATH"] = str(PROJECT_ROOT / "training")

if not IN_COLAB:
    subprocess.check_call(
        [sys.executable, "training/scripts/export_android_assets.py"],
        cwd=str(PROJECT_ROOT),
    )
    asset = (
        PROJECT_ROOT
        / "android"
        / "classifier-sdk"
        / "src"
        / "main"
        / "assets"
        / "model"
        / "sms_bytecnn_int8.tflite"
    )
    print("Exported:", asset, "exists=", asset.is_file())
else:
    export_dir = Path("/content/colab_export")
    if export_dir.exists():
        shutil.rmtree(export_dir)
    export_dir.mkdir(parents=True)
    shutil.copytree(PROJECT_ROOT / "training" / "artifacts", export_dir / "artifacts")
    metrics = PROJECT_ROOT / "training" / "reports" / "metrics"
    if metrics.exists():
        shutil.copytree(metrics, export_dir / "metrics")
    manifests = PROJECT_ROOT / "training" / "data" / "manifests"
    if manifests.exists():
        shutil.copytree(manifests, export_dir / "manifests")
    tflite = PROJECT_ROOT / "training" / "artifacts" / "student" / "sms_bytecnn_int8.tflite"
    if tflite.exists():
        shutil.copy2(tflite, export_dir / tflite.name)

    zip_path = Path("/content/colab_export.zip")
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for f in export_dir.rglob("*"):
            if f.is_file():
                zf.write(f, f.relative_to(export_dir.parent))
    print("Wrote", zip_path)

    try:
        from google.colab import files

        files.download(str(zip_path))
    except Exception as exc:  # noqa: BLE001
        print("自动下载失败（VS Code Colab 有时无 files.download）：", exc)
        print("请在文件浏览器打开", zip_path, "手动下载，再拷回本机 training/")

## 完成后

**本机内核：** `export_android_assets` 已写入 SDK，可直接编 APK。

**Colab 内核：** 把 `colab_export.zip` 解压后覆盖本机：

```powershell
Copy-Item -Recurse -Force .\colab_export\artifacts\* training\artifacts\
Copy-Item -Recurse -Force .\colab_export\metrics\* training\reports\metrics\
Copy-Item -Recurse -Force .\colab_export\manifests\* training\data\manifests\
$env:PYTHONPATH = "training"
python training\scripts\export_android_assets.py
```

Checklist：`teacher_logits_manifest.json` 存在 · `used_distillation=true` · 未用合成指标宣称 ≥98%。